In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# df1 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260211.parquet")
# df2 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260212.parquet")
# df3 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260213.parquet")
# df4 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260214.parquet")
# df5 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260215.parquet")
# df6 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260216.parquet")
# df7 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260217.parquet")
# df8 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260218.parquet")
# df9 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260219.parquet")


from pathlib import Path
import re
import pandas as pd
import pyarrow.dataset as ds

BASE = Path(r"C:\msys64\home\for\10th\00_Project\04_final\02_parquet_file")  # 내 경로로 수정
MAP = "Erangel"
PLATFORM = "kakao"
DATE_FROM = "20260211"
DATE_TO   = "20260219"

# 1) 대상 parquet 파일만 골라오기
pat = re.compile(rf"^{MAP}_telemetry_{PLATFORM}_(\d{{8}})\.parquet$")
files = []
for fp in sorted(BASE.glob(f"{MAP}_telemetry_{PLATFORM}_*.parquet")):
    m = pat.match(fp.name)
    if not m:
        continue
    yyyymmdd = m.group(1)
    if DATE_FROM <= yyyymmdd <= DATE_TO:
        files.append(fp)

print("files:", len(files))
if not files:
    raise SystemExit("No parquet files matched. 패턴/날짜/경로 확인 필요")

# 2) 출력 폴더 준비
OUTDIR = BASE / f"csv_{MAP}_{PLATFORM}_{DATE_FROM}_{DATE_TO}"
OUTDIR.mkdir(parents=True, exist_ok=True)

EVENTS = [
    "LogPlayerAttack",
    "LogPlayerMakeGroggy",
    "LogPlayerTakeDamage",
    "LogPlayerKillV2",
    "LogPlayerPosition",
    "LogItemPickup",
    "LogMatchStart",
    "LogMatchEnd",
    "LogPhaseChange",
]

# 3) 스키마 기반으로 실제 존재 컬럼만 쓰도록 처리(중요: 컬럼명 불일치로 에러 방지)
dataset = ds.dataset([str(p) for p in files], format="parquet")
all_cols = set(dataset.schema.names)

# 최소 공통 컬럼 (너 데이터에 맞게 필요하면 추가)
BASE_COLS = ["_D", "_T", "matchId"]  # accountId가 있다면 추가 가능
use_cols = [c for c in BASE_COLS if c in all_cols]

# 너무 많이 읽고 싶으면 아래처럼 "필요한 컬럼"을 더 추가해도 됨 (존재하는 것만)
CANDIDATES = [
    "accountId", "playerId", "attacker", "victim", "damage", "damageDealt",
    "killer", "killed", "weapon", "item", "itemId", "itemName",
    "x", "y", "z", "location", "character", "teamId",
    "time", "timestamp"
]
use_cols += [c for c in CANDIDATES if c in all_cols]
use_cols = list(dict.fromkeys(use_cols))  # 중복 제거, 순서 유지

print("use_cols:", use_cols)

# 4) 배치로 스캔 (전체 1-pass) 후 _T로 분리 저장
scanner = dataset.scanner(columns=use_cols, batch_size=65536)

# 헤더 1회만 쓰기 위한 플래그
written = {ev: False for ev in EVENTS}

for batch in scanner.to_batches():
    df = batch.to_pandas()

    # _T 컬럼이 없으면 작업 불가
    if "_T" not in df.columns:
        raise RuntimeError("_T column not found in loaded batch. 스키마/컬럼명 확인 필요")

    # 관심 이벤트만 필터
    df = df[df["_T"].isin(EVENTS)]
    if df.empty:
        continue

    # 이벤트별로 쪼개서 append 저장
    for ev, sub in df.groupby("_T"):
        out = OUTDIR / f"{MAP}_{PLATFORM}_{DATE_FROM}_{DATE_TO}__{ev}.csv"
        sub.to_csv(out, mode="a", index=False, header=(not written[ev]))
        written[ev] = True

print("done. output dir:", OUTDIR)
print("written:", {k:v for k,v in written.items() if v})


files: 9
use_cols: ['_D', '_T', 'matchId', 'accountId', 'attacker', 'damage', 'killer']


PermissionError: [Errno 13] Permission denied: 'C:\\msys64\\home\\for\\10th\\00_Project\\04_final\\02_parquet_file\\csv_Erangel_kakao_20260211_20260219\\Erangel_kakao_20260211_20260219__LogItemPickup.csv'

In [3]:
from pathlib import Path
import pyarrow.dataset as ds
import pandas as pd

# ===== 설정 =====
PARQUET_PATH = Path(r"C:\msys64\home\for\10th\00_Project\04_final\02_parquet_file\Erangel_telemetry_kakao_20260211.parquet")
EVENT_NAME = "LogPlayerAttack"   # 예: "LogPlayerTakeDamage", "LogPlayerKillV2" 등
BATCH_SIZE = 65536

# ===== 로드 (pyarrow.dataset) =====
dataset = ds.dataset(str(PARQUET_PATH), format="parquet")
all_cols = dataset.schema.names

# _T 컬럼이 반드시 있어야 이벤트 필터 가능
if "_T" not in all_cols:
    raise RuntimeError("이 parquet에 _T 컬럼이 없습니다. 컬럼명 확인 필요.")

# 결과 누적용: 각 컬럼이 'non-null 값이 한 번이라도 있었는지'
has_value = {c: False for c in all_cols}
rows_seen = 0
rows_matched = 0

scanner = dataset.scanner(columns=all_cols, batch_size=BATCH_SIZE)

for batch in scanner.to_batches():
    df = batch.to_pandas()
    rows_seen += len(df)

    # 이벤트 필터
    sub = df[df["_T"] == EVENT_NAME]
    if sub.empty:
        continue

    rows_matched += len(sub)

    # 컬럼별로 non-null 존재 여부 체크
    # (이미 True 된 컬럼은 스킵해서 속도 조금 개선)
    for c in all_cols:
        if has_value[c]:
            continue
        # pandas 기준: NaN/None 모두 notna()에서 False 처리
        if sub[c].notna().any():
            has_value[c] = True

print(f"전체 행: {rows_seen:,}")
print(f"{EVENT_NAME} 행: {rows_matched:,}")

non_null_cols = [c for c, v in has_value.items() if v]
print(f"\n[{EVENT_NAME}]에서 값이 존재하는 컬럼 수: {len(non_null_cols)}")
print(non_null_cols)


전체 행: 4,417,623
LogPlayerAttack 행: 664,675

[LogPlayerAttack]에서 값이 존재하는 컬럼 수: 42
['matchId', '_D', '_T', 'common_isGame', 'attackId', 'fireWeaponStackCount', 'attacker_name', 'attacker_teamId', 'attacker_health', 'attacker_location_x', 'attacker_location_y', 'attacker_location_z', 'attacker_ranking', 'attacker_individualRanking', 'attacker_accountId', 'attacker_isInBlueZone', 'attacker_isInRedZone', 'attacker_inSpecialZone', 'attacker_isInVehicle', 'attacker_zone', 'attacker_type', 'attacker_isDBNO', 'attackType', 'weapon_itemId', 'weapon_stackCount', 'weapon_category', 'weapon_subCategory', 'weapon_attachedItems', 'vehicle_vehicleType', 'vehicle_vehicleId', 'vehicle_seatIndex', 'vehicle_healthPercent', 'vehicle_feulPercent', 'vehicle_altitudeAbs', 'vehicle_altitudeRel', 'vehicle_velocity', 'vehicle_isWheelsInAir', 'vehicle_isInWaterVolume', 'vehicle_isEngineOn', 'vehicle_location_x', 'vehicle_location_y', 'vehicle_location_z']


In [4]:
from pathlib import Path
import pyarrow.dataset as ds
import pandas as pd

PARQUET_PATH = Path(r"C:\msys64\home\for\10th\00_Project\04_final\02_parquet_file\Erangel_telemetry_kakao_20260211.parquet")
BATCH_SIZE = 65536

EVENTS = [
    "LogParachuteLanding",
    "LogPlayerPosition",
    "LogMatchStart",
    "LogGameStatePeriodic",
    "LogVehicleRide",
    "LogVehicleLeave",
    "LogPhaseChange"
]

dataset = ds.dataset(str(PARQUET_PATH), format="parquet")
all_cols = dataset.schema.names

if "_T" not in all_cols:
    raise RuntimeError("이 parquet에 _T 컬럼이 없습니다. 컬럼명 확인 필요.")

# 이벤트별로: 컬럼이 한 번이라도 non-null이었는지 저장
has_value = {ev: {c: False for c in all_cols} for ev in EVENTS}
rows_seen = 0
rows_matched = {ev: 0 for ev in EVENTS}

scanner = dataset.scanner(columns=all_cols, batch_size=BATCH_SIZE)

for batch in scanner.to_batches():
    df = batch.to_pandas()
    rows_seen += len(df)

    # 관심 이벤트만 남기기
    df = df[df["_T"].isin(EVENTS)]
    if df.empty:
        continue

    # 이벤트별로 처리
    for ev, sub in df.groupby("_T"):
        rows_matched[ev] += len(sub)

        # 이미 True인 컬럼은 스킵
        hv = has_value[ev]
        for c in all_cols:
            if hv[c]:
                continue
            if sub[c].notna().any():
                hv[c] = True

print(f"전체 행: {rows_seen:,}")
print("========================================")

for ev in EVENTS:
    non_null_cols = [c for c, v in has_value[ev].items() if v]
    print(f"\n[{ev}]")
    print(f"  행 수: {rows_matched[ev]:,}")
    print(f"  값이 존재하는 컬럼 수: {len(non_null_cols)}")
    print(f"  컬럼: {non_null_cols}")


전체 행: 4,417,623

[LogParachuteLanding]
  행 수: 10,170
  값이 존재하는 컬럼 수: 21
  컬럼: ['matchId', '_D', '_T', 'common_isGame', 'character_name', 'character_teamId', 'character_health', 'character_location_x', 'character_location_y', 'character_location_z', 'character_ranking', 'character_individualRanking', 'character_accountId', 'character_isInBlueZone', 'character_isInRedZone', 'character_inSpecialZone', 'character_isInVehicle', 'character_zone', 'character_type', 'character_isDBNO', 'distance']

[LogPlayerPosition]
  행 수: 773,989
  값이 존재하는 컬럼 수: 36
  컬럼: ['matchId', '_D', '_T', 'common_isGame', 'character_name', 'character_teamId', 'character_health', 'character_location_x', 'character_location_y', 'character_location_z', 'character_ranking', 'character_individualRanking', 'character_accountId', 'character_isInBlueZone', 'character_isInRedZone', 'character_inSpecialZone', 'character_isInVehicle', 'character_zone', 'character_type', 'character_isDBNO', 'elapsedTime', 'numAlivePlayers', 'vehi

In [5]:
import pyarrow.dataset as ds
import pandas as pd
from pathlib import Path

PARQUET_PATH = Path(r"C:\msys64\home\for\10th\00_Project\04_final\02_parquet_file\Erangel_telemetry_kakao_20260211.parquet")
dataset = ds.dataset(str(PARQUET_PATH), format="parquet")

scanner = dataset.scanner(columns=["_T"], batch_size=200000)
types = set()
for b in scanner.to_batches():
    s = pd.Series(b.to_pandas()["_T"]).dropna().unique().tolist()
    types.update(s)

# GameState/Parachute 같은 것만 필터해서 보기
keywords = ["GameState", "Parachute", "Landing", "Plane", "Flight", "Zone", "Circle"]
filtered = sorted([t for t in types if any(k in t for k in keywords)])
print("Filtered _T:", filtered)
print("Total _T count:", len(types))

Filtered _T: ['LogGameStatePeriodic', 'LogParachuteLanding', 'LogSpecialZoneInCharacters']
Total _T count: 47
